# Setup
データ取り込みを行う。 PostgresSQL PgVector 想定。

In [ ]:
import os
import dotenv

dotenv.load_dotenv()
ENV_PG_CONNECTION_STRING = os.getenv("ENV_PG_CONNECTION_STRING")
ENV_GEMINI_API_KEY = os.getenv("ENV_GEMINI_API_KEY")

In [ ]:
from langchain_postgres import PGEngine
from sqlalchemy import create_engine

# エンジン初期化
assert ENV_PG_CONNECTION_STRING is not None
assert ENV_GEMINI_API_KEY is not None

sa_engine = create_engine(ENV_PG_CONNECTION_STRING)
pg_engine = PGEngine.from_connection_string(ENV_PG_CONNECTION_STRING, pool_size=5)

# Extract

In [ ]:
from pathlib import Path
from llama_index.core import SimpleDirectoryReader
from assistant_agent.loaders import MarkdownReader

# Vault から読み込み
vault_path = Path("../../docs/dataset_website").resolve()
loader = SimpleDirectoryReader(
    input_dir=vault_path,
    recursive=True,
    file_extractor={".md": MarkdownReader()}
)
docs = loader.load_data()

print(f"{len(docs)} 件のノートを読み込みました")

# Load

In [ ]:
from assistant_agent.entities.base import VaultUtils
from assistant_agent.entities.postgres import VaultBase, SampleEntity

# DB へ取り込み
VaultBase.metadata.create_all(sa_engine)
VaultUtils.sync(docs[:10], sa_engine, SampleEntity)

print(f"{len(docs)} 件のノートを読み込みました")

In [ ]:
import sqlalchemy
from assistant_agent.entities.postgres import SampleEntity

with sa_engine.connect() as sess:
    res = sess.execute(sqlalchemy.select(SampleEntity).limit(10))

# res = [len(item[2]) for item in res]
list(res)

In [ ]:
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TransformComponent

chunk_size = 800
chunk_overlap = 80
JAPANESE_PARAGRAPH_SEP = "\n\n"

trans: list[TransformComponent] = [
    SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        paragraph_separator=JAPANESE_PARAGRAPH_SEP,
    ),
]
pipe = IngestionPipeline(transformations=trans)
res = pipe.run(documents=docs[:2])
print("\n=====================\n".join([item.text for item in res]))  # pyright: ignore[reportAttributeAccessIssue]


In [ ]:
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from assistant_agent.utils.store_context import PostgresStoreContext
from assistant_agent.entities.postgres import SampleEntity
from assistant_agent.services import VaultSampleRetriever

assert ENV_PG_CONNECTION_STRING is not None
assert ENV_GEMINI_API_KEY is not None

store_ctx = PostgresStoreContext(ENV_PG_CONNECTION_STRING, schema_name="app")
sa_engine = store_ctx.get_engine()
sample_retriever = VaultSampleRetriever(
    "sample_docstore",
    "sample_vectors",
    store_context=store_ctx,
    transformations=trans,
    embed_model=GoogleGenAIEmbedding(
        model_name="gemini-embedding-001",
        api_key=ENV_GEMINI_API_KEY,
    ),
    embed_dim=3072,
    vault_entity=SampleEntity,
)

In [ ]:
# sample_retriever.sync_chunks()

# Retrieval

In [ ]:
query_res = sample_retriever.search_documents("BPM", 5)
for item in query_res:
    print(f"{item.score=}")  # pyright: ignore[reportAttributeAccessIssue]
    print(item.text)
    print("==================")